John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

Import Packages

In [1]:
# install.packages("tidymodels")
# install.packages('rlang')
# install.packages("ranger")
# install.packages("rpart.plot")
# install.packages("vip")
library(tidyverse)
library(tidymodels)
library(rpart.plot) # for plotting decision tree
library(vip) # for ranking feature importance
# library(ranger) # engine for random forest

# set random seed
set.seed(42)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.5.2
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
── Attaching packages ────────────────────────────────────── tidymodels 1.4.1 ──

✔ broom        1.0.10     ✔ rsample      1.3.1 
✔ dials        1.4.2      ✔ tailor       0.1.0 
✔ infer        1.0.9      ✔ tune         2.0.0 
✔ modeldata    1.5.1      ✔ workflows    1.3.0 
✔ parsnip      1.3.3      ✔ workflowsets 1.1.1 
✔ recipes      1.3.1      ✔ yardstick    1.3.2 

── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() mask

Load data

In [ ]:
data <- read_csv("data/matching_opportunities.csv")

Rows: 709 Columns: 82
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (81): UST, time_rel_sunrise, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
dttm  (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
# Model Inputs
model_columns <- c(
        "UST", 
        # "does_it_match",
        # "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        "M1_1_last_UST", "M2_1_last_UST", "M3_1_last_UST", "M4_1_last_UST", "M5_1_last_UST", "M6_1_last_UST",
        # "time_M1_last_song", "time_M2_last_song", "time_M3_last_song", "time_M4_last_song", "time_M5_last_song", "time_M6_last_song",
    	"time_since_M1_1_last_song", "time_since_M2_1_last_song", "time_since_M3_1_last_song", "time_since_M4_1_last_song", "time_since_M5_1_last_song", "time_since_M6_1_last_song"
    )

In [4]:
# Set data types and select input columns
data <- data %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) %>%
    select(all_of(model_columns)) %>%
    drop_na() # drop any row containing at least one NA value

In [5]:
glimpse(data)

Rows: 428
Columns: 13
$ UST                       <fct> 20, 3, 0, 13, 29, 9, 8, 3, 116, 29, 10, 8, 3…
$ M1_1_last_UST             <fct> 0, 0, 3, 0, 8, 29, 9, 8, 3, 116, 29, 10, 8, …
$ M2_1_last_UST             <fct> 0, 8, 9, 0, 0, 5, 16, 9, 3, 8, 8, 9, 3, 9, 1…
$ M3_1_last_UST             <fct> 0, 8, 0, 0, 9, 8, 8, 0, 0, 0, 8, 9, 3, 0, 16…
$ M4_1_last_UST             <fct> 0, 9, 9, 0, 9, 3, 3, 8, 8, 9, 9, 10, 10, 3, …
$ M5_1_last_UST             <fct> 0, 8, 0, 0, 0, 0, 8, 0, 0, 10, 10, 8, 8, 20,…
$ M6_1_last_UST             <fct> 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 3, 0, 0,…
$ time_since_M1_1_last_song <dbl> 5.890881, 43.989810, 14.397677, 55.771537, 8…
$ time_since_M2_1_last_song <dbl> 9.8429906, 5.8120005, 5.5579771, 5.1593139, …
$ time_since_M3_1_last_song <dbl> 6.117981, 7.213702, 8.059989, 4.721791, 8.57…
$ time_since_M4_1_last_song <dbl> 2.010357, 8.070434, 5.797139, 4.745750, 9.39…
$ time_since_M5_1_last_song <dbl> 5.75153467, 3.51400991, 0.08138158, 9.151744…
$ time_since_M6_1_

In [6]:

# train test split
train_test_split <- initial_split(data, prop = 0.7, strata=UST)

train_data <- train_test_split %>% training()
test_data <- train_test_split %>% testing()

# In training data, how many of each song type?
n_ust <- train_data %>%
    group_by(UST) %>%
    summarize(n = n()) %>%
    mutate(percent = 100*n/nrow(train_data)) %>%
    arrange(percent)
n_ust

# use v-fold cross validation to assess model accuracy
v <- 5
train_vfold <- vfold_cv(train_data, v=v, strata = UST)

UST,n,percent
<fct>,<int>,<dbl>
16,1,0.3367003
118,1,0.3367003
119,1,0.3367003
12,2,0.6734007
116,2,0.6734007
117,2,0.6734007
0,4,1.3468013
5,16,5.3872054
17,20,6.7340067


In [7]:
# Data preprocessing using a recipe
recipe <- recipe(UST ~ ., data = train_data) %>%
    step_zv(all_predictors()) %>%
  step_novel(all_nominal_predictors()) %>%
  step_unknown(all_nominal_predictors()) %>%
  step_dummy(all_nominal_predictors(), one_hot = TRUE) %>%
  step_normalize(all_numeric_predictors()) %>%
    step_zv(all_predictors())

In [8]:
# Create model specification
model_tuning_spec <- decision_tree(tree_depth = tune(), min_n=tune()) %>%
    set_engine("rpart") %>%
    set_mode("classification")

In [9]:
tune_grid <- tidyr::crossing(
    tree_depth = c(10, 20, 30), # The number of variables randomly sampled as candidates at each split
    min_n = c(10, 20, 30, 40, 50) # Minimum number of data points in a node required for a split
)

model_tuning <- workflow() %>%
    add_recipe(recipe) %>%
    add_model(model_tuning_spec) %>%

        tune_grid(resamples = train_vfold, grid=tune_grid) # for v-fold cross validation, tuning k parameter
        # fit(data = train_data) # to train on whole train set with no validation set

model_tuning

→ A | warning: !  The following columns have zero variance so scaling cannot be used:
                 M1_1_last_UST_X16, M1_1_last_UST_X115, M1_1_last_UST_X117,
                 M1_1_last_UST_X118, M1_1_last_UST_X119, M1_1_last_UST_X120,
                 M1_1_last_UST_new, M1_1_last_UST_unknown, M2_1_last_UST_new,
                 M2_1_last_UST_unknown, M3_1_last_UST_new, M3_1_last_UST_unknown,
                 M4_1_last_UST_new, M4_1_last_UST_unknown, M5_1_last_UST_new,
                 M5_1_last_UST_unknown, M6_1_last_UST_X17, M6_1_last_UST_new, and
                 M6_1_last_UST_unknown.
               ℹ Consider using ?step_zv (`?recipes::step_zv()`) to remove those columns
                 before normalizing.

There were issues with some computations   A: x1

→ B | error:   subscript out of bounds

There were issues with some computations   A: x1
→ C | warning: !  The following columns have zero variance so scaling cannot be used:
                 M1_1_last_UST_X16, M1_1_last_UST

In [10]:
# Accuracies listed use 5-fold cross validation 2025-10-17
model_tuning_accuracies <- model_tuning %>% collect_metrics() %>% filter(.metric == "accuracy")
model_tuning_accuracies

ERROR: [1m[33mError[39m in `estimate_tune_results()`:[22m
[1m[22m[33m![39m All models failed. Run `show_notes(.Last.tune.result)` for more
  information.


In [ ]:
show_notes(.Last.tune.result)

In [ ]:
# plot accuracies of all values of hyperparameters

ggplot(model_tuning_accuracies) +
    geom_tile(aes(x=tree_depth, y=min_n, fill=mean)) +
    geom_text(aes(x=tree_depth, y=min_n, label=round(mean, 2), size=5)) +
    labs(x = "Randomly Sampled Variables", y = "Minimum Samples to Split", color="Estimated Accuracy", title='Accuracy') +
    theme(text = element_text(size=20)) +
    scale_fill_gradient(low="white", high="red")

# Print best hyperparameter values
best_params <- model_tuning_accuracies %>% arrange(desc(mean)) %>% head(1) %>% pull(tree_depth, min_n)
print("Best hyperparameters: (mtry, min_n)")
print(best_params)

In [ ]:
# retrain model on best parameters
best_params <- model_tuning %>%
  select_best()
print("Best Parameters:")
print(best_params)

model <- workflow() %>%
    add_recipe(recipe) %>%
    add_model(model_tuning_spec) %>%
    finalize_workflow(best_params) %>%
    last_fit(train_test_split)

# model

In [ ]:
model %>% extract_fit_engine() %>% vip(num_features=20)

In [ ]:
png("ust_tree.png", width = 1600, height = 800, res = 300)
model %>% extract_fit_engine() %>% rpart.plot(extra = 101, type=3, tweak=0.75)
dev.off() # Close the PNG device

In [ ]:
# Predict on the test set, and create a new DataFrame with inputs and outputs
test_predictions <- model %>% extract_workflow() %>% predict(test_data) %>%
    bind_cols(test_data)

In [ ]:
# test_predictions

In [ ]:
# What was the accuracy, precision, and recall of our model?
test_accuracy <- test_predictions %>% metrics(truth = UST, estimate = .pred_class) %>% filter(.metric == "accuracy")
test_accuracy

test_predictions %>% group_by(UST) %>% metrics(truth = UST, estimate = .pred_class) %>% filter(.metric == "accuracy") %>% arrange(desc(.estimate)) %>% 
    select(UST, .estimate) %>% rename(estimated_accuracy = .estimate) %>%
    left_join(n_ust %>% select(UST, n), by = "UST") 

test_predictions %>% group_by(UST) %>% precision(truth = UST, estimate = .pred_class, event_level = "multiclass") %>% arrange(desc(.estimate)) # Precision = correct positive predictions / total positive predictions
test_predictions %>% group_by(UST) %>% recall(truth = UST, estimate = .pred_class, event_level = "multiclass") %>% arrange(desc(.estimate)) # Recall = correct positive predictions / total positive test set observations

In [ ]:
test_predictions %>% pull(UST) %>% levels()
confusion_matrix <- test_predictions %>%
    conf_mat(truth = UST, estimate = .pred_class)
confusion_matrix